# 00 Setup: calibrate a clip and plot players on the pitch

Installs the sn-gamestate baseline, downloads one free SoccerNet GSR clip, then runs YOLO + ByteTrack + PnLCalib on its first 100 frames and draws each player's pitch position on a radar view. Use a GPU runtime (Runtime > Change runtime type): this takes about a minute on an A100 and over an hour on CPU.

In [ ]:
!git clone -q https://github.com/ttkgas/throughball.git
%cd throughball

In [ ]:
!bash setup.sh

In [ ]:
!third_party/sn-gamestate/.venv/bin/python scripts/get_clip.py SNGS-021

In [ ]:
# Colab's inline plot backend doesn't exist in the 3.9 env, so use Agg there.
!MPLBACKEND=Agg third_party/sn-gamestate/.venv/bin/tracklab --config-dir configs -cn throughball_setup

Left radar: our predictions. Right radar: ground truth. The full video is under `outputs/`.

In [ ]:
import glob, cv2, matplotlib.pyplot as plt

video = sorted(glob.glob("outputs/throughball-setup/**/*.mp4", recursive=True))[-1]
cap = cv2.VideoCapture(video)
cap.set(cv2.CAP_PROP_POS_FRAMES, cap.get(cv2.CAP_PROP_FRAME_COUNT) // 2)
ok, frame = cap.read()
plt.figure(figsize=(16, 9))
plt.imshow(frame[..., ::-1])
plt.axis("off")
plt.title(video)

Stage 1 output: per-frame cameras (`cameras.json`) and player pitch positions in meters (`players.csv`) under `outputs/stage1/`. Stage 3 reads these.

In [ ]:
!third_party/sn-gamestate/.venv/bin/python scripts/export_stage1.py
!head -4 outputs/stage1/*/players.csv